# PyTorch Gradient Reset Methods

This notebook explains why gradients must be reset and compares the main ways to do it in PyTorch.

What this notebook covers:
1. Why gradients accumulate by default.
2. `optimizer.zero_grad()`.
3. `param.grad.zero_()`.
4. `param.grad = None`.
5. `optimizer.zero_grad(set_to_none=True)`.
6. A small test showing how `None` and zero tensors behave differently when a parameter is unused.

Run the cells from top to bottom and compare both the printed values and the reported gradient states.

In [1]:
import torch
import torch.nn as nn

## Test 1: Gradients accumulate if you do not reset them

PyTorch adds new gradients into the existing `.grad` buffer.

If you call `.backward()` twice without resetting gradients, the second gradient is added to the first one. This is useful for gradient accumulation, but it is a bug in a standard training loop if done accidentally.

In [2]:
# Test 1: accumulation behavior
w = torch.tensor(2.0, requires_grad=True)

loss1 = w ** 2
loss1.backward()
print("After first backward, grad =", w.grad.item())  # 2*w = 4

loss2 = w ** 2
loss2.backward()
print("After second backward without reset, grad =", w.grad.item())  # 8

# Reset manually for the next tests
w.grad = None
print("After reset, grad =", w.grad)

After first backward, grad = 4.0
After second backward without reset, grad = 8.0
After reset, grad = None


## Test 2: Compare the main reset methods

This section checks the state of `.grad` after each reset strategy.

Interpretation:
- A printed tensor means the gradient buffer still exists.
- `None` means the gradient buffer has been removed and will be recreated on the next backward pass.

Important note: the behavior of plain `optimizer.zero_grad()` can depend on the PyTorch version and defaults. To avoid ambiguity, this notebook uses both `set_to_none=False` and `set_to_none=True` explicitly.

In [ ]:
# Test 2: compare reset methods on a simple parameter
p = nn.Parameter(torch.tensor(3.0))
optimizer = torch.optim.SGD([p], lr=0.1)

# Build one gradient
loss = p ** 2
loss.backward()
print("Initial grad:", p.grad)

# Method 1: zero_()
p.grad.zero_()
print("After p.grad.zero_():", p.grad)

# Rebuild gradient
loss = p ** 2
loss.backward()
print("After another backward:", p.grad)

# Method 2: p.grad = None
p.grad = None
print("After p.grad = None:", p.grad)

# Rebuild gradient
loss = p ** 2
loss.backward()
print("After backward from None:", p.grad)

# Method 3: optimizer.zero_grad()
optimizer.zero_grad()
print("After optimizer.zero_grad():", p.grad)

# Rebuild gradient
loss = p ** 2
loss.backward()

# Method 4: optimizer.zero_grad(set_to_none=True)
optimizer.zero_grad(set_to_none=True)
print("After optimizer.zero_grad(set_to_none=True):", p.grad)

Initial grad: tensor(6.)
After p.grad.zero_(): tensor(0.)
After another backward: tensor(6.)
After p.grad = None: None
After backward from None: tensor(6.)
After optimizer.zero_grad(): None
After optimizer.zero_grad(set_to_none=True): None


## Test 3: Unused parameter behavior

This is the subtle case that often matters in debugging.

If a parameter is not used in the forward pass:
- with `grad = None`, it stays `None`
- with a zero tensor already present, it stays a zero tensor

That difference can be useful if you want to detect whether a parameter actually participated in the computation graph.

In [4]:
# Test 3: unused parameter behavior
used = nn.Parameter(torch.tensor(2.0))
unused = nn.Parameter(torch.tensor(5.0))

# Case A: start from None
used.grad = None
unused.grad = None
loss = used ** 2
loss.backward()
print("Case A - used.grad:", used.grad)
print("Case A - unused.grad:", unused.grad)

# Case B: unused parameter already has a zero tensor grad
used.grad = None
unused.grad = torch.zeros_like(unused)
loss = used ** 2
loss.backward()
print("Case B - used.grad:", used.grad)
print("Case B - unused.grad:", unused.grad)

Case A - used.grad: tensor(4.)
Case A - unused.grad: None
Case B - used.grad: tensor(4.)
Case B - unused.grad: tensor(0.)


## Takeaway

Use these rules in practice:

1. Use `optimizer.zero_grad()` as the default training-loop pattern.
2. Use `optimizer.zero_grad(set_to_none=True)` when you want slightly better memory and speed behavior.
3. Use `param.grad.zero_()` only for manual or debugging scenarios.
4. Remember that `None` and zero are not the same state when a parameter is unused.